## 머신러닝응용 5주차 과제

1. 아래 Linear Regression 모델의 Feature를 조정하여 성능을 향상 → 결과를 출력하고 마크다운 셀로 설명하세요.

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 데이터
df = pd.read_csv('https://raw.githubusercontent.com/tysep16/26_02_ML/refs/heads/main/SeoulBike.csv')
# 전처리
df.columns = ['date', 'rented', 'hour', 'temp', 'humidity', 'wind', 'visibility', 'dew', 'solar', 'rain', 'snow', 'season', 'holiday', 'functioning']
df = df[df.functioning == 'Yes']
df.holiday = (df.holiday == 'Holiday').astype(int)
df = df.drop(['date', 'functioning'], axis=1)
y = df['rented']
features = df.drop('rented', axis=1)
features = pd.get_dummies(features, columns=['hour', 'season'], drop_first=True, dtype=int)

# 평가
def evaluate(model, X_test, y_test):
    pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred) ** 0.5
    r2 = r2_score(y_test, pred)
    print('MAE :', round(mae, 1), ' RMSE :', round(rmse, 1), ' R2 :', round(r2, 3))
    return pred

# Feature Engineering
# X3
X3 = features[['temp', 'humidity', 'wind', 'solar', 'rain', 'snow', 'season_Spring', 'season_Summer', 'season_Winter']]
X_train, X_test, y_train, y_test = train_test_split(X3, y, test_size=0.2, random_state=42)
lr3 = LinearRegression()
lr3.fit(X_train, y_train)
pred3 = evaluate(lr3, X_test, y_test)

# X4
hour_cols = [c for c in features.columns if c.startswith('hour_')]
X4 = features[list(X3.columns) + ['holiday'] + hour_cols]
X_train, X_test, y_train, y_test = train_test_split(X4, y, test_size=0.2, random_state=42)
lr4 = LinearRegression()
lr4.fit(X_train, y_train)
pred4 = evaluate(lr4, X_test, y_test)

# X5
features['temp2'] = features['temp'] ** 2
features['humid2'] = features['humidity'] ** 2
features['is_rain'] = (features['rain'] > 0).astype(int)

X5 = features[list(X4.columns) + ['temp2', 'humid2', 'is_rain']]
X_train, X_test, y_train, y_test = train_test_split(X5, y, test_size=0.2, random_state=42)
lr5 = LinearRegression()
lr5.fit(X_train, y_train)
pred5 = evaluate(lr5, X_test, y_test)

# X6
hour_temp_cols = []
for h in hour_cols:
    features[h + '_temp'] = features[h] * features['temp']
    hour_temp_cols.append(h + '_temp')

X6 = features[list(X5.columns) + hour_temp_cols]
X_train, X_test, y_train, y_test = train_test_split(X6, y, test_size=0.2, random_state=42)
lr6 = LinearRegression()
lr6.fit(X_train, y_train)
pred6 = evaluate(lr6, X_test, y_test)

MAE : 340.2  RMSE : 458.0  R2 : 0.466
MAE : 274.3  RMSE : 363.2  R2 : 0.664
MAE : 268.8  RMSE : 353.0  R2 : 0.683
MAE : 219.8  RMSE : 302.8  R2 : 0.766


### 결과 분석

| 모델 | 추가한 Feature | MAE | RMSE | R² |
|---|---|---|---|---|
| X3 (기존) | 기온, 습도, 풍속, 일사량, 강수량, 적설량, 계절 | 340.2 | 458.0 | 0.466 |
| X4 | + 공휴일, 시간대(`hour_*`) | 274.3 | 363.2 | 0.664 |
| X5 | + 기온², 습도², 비 여부 | 268.8 | 353.0 | 0.683 |
| X6 | + 시간대 × 기온 | **219.8** | **302.8** | **0.766** |

- **X4**: 시간대별 수요 차이를 반영하기 위해 시간대와 공휴일을 추가, R²가 가장 크게 상승
- **X5**: 직선으로는 표현하기 어려운 곡선 관계를 반영하기 위해 기온², 습도²을 추가, 비 여부 변수도 함께 포함
- **X6**: 시간대에 따라 기온이 미치는 영향이 다를 수 있으므로 시간대 x 기온을 추가

**결론**: R²는 0.466 → 0.766, MAE는 340.2 → 219.8로 개선
